# Phase 3b: NFCorpus Medical Reranking

AutoDL GPU — 医疗信息检索段落重排序实验（跨领域验证）

| # | 实验 | 方法 | 数据集 | 模型 |
|---|------|------|--------|------|
| E21 | LoRA NFCorpus Qwen    | LoRA (r=16) | NFCorpus Rerank | Qwen2.5-1.5B |
| E22 | LoRA NFCorpus Llama   | LoRA (r=16) | NFCorpus Rerank | Llama-3.2-1B |
| E23 | Full FT NFCorpus Qwen | Full FT     | NFCorpus Rerank | Qwen2.5-1.5B |
| E24 | Full FT NFCorpus Llama| Full FT     | NFCorpus Rerank | Llama-3.2-1B |
| E25 | Random NFCorpus Qwen  | LoRA (shuffled labels) | NFCorpus Rerank | Qwen2.5-1.5B |
| E26 | Random NFCorpus Llama | LoRA (shuffled labels) | NFCorpus Rerank | Llama-3.2-1B |

## 0. 环境准备

In [ ]:
import os, sys
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ['WANDB_MODE'] = 'offline'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)

# 软链到数据盘
for target, link in [('/root/autodl-tmp/outputs', '/root/MLP/outputs'),
                      ('/root/autodl-tmp/logs',    '/root/MLP/logs'),
                      ('/root/autodl-tmp/data',    '/root/MLP/data')]:
    if not os.path.islink(link):
        if os.path.isdir(link):
            print(f'  ⚠ {link} is a real dir, skipping')
        else:
            os.symlink(target, link)
            print(f'  ✓ created: {link} -> {target}')
    else:
        print(f'  ✓ symlink exists: {link} -> {os.readlink(link)}')

!pwd && ls
print(f'Python: {sys.executable}')
print(f'HF_HOME: {os.environ["HF_HOME"]}')
print(f'HF_ENDPOINT: {os.environ["HF_ENDPOINT"]}')
print(f'WANDB_MODE: {os.environ["WANDB_MODE"]}')

In [ ]:
import sys
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub datasets
print('\n✅ 安装完成！如果是第一次装，请重启内核：Kernel → Restart Kernel')

In [ ]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# HuggingFace 登录 (Llama 需要)
from huggingface_hub import login
login()
print('HuggingFace 登录成功')

## 1. 数据下载与格式化

从 HuggingFace 下载 `BeIR/nfcorpus`（医疗信息检索），构建 5 候选段落重排序任务数据。

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/data/nfcorpus/nfcorpus_formatting.py --output_dir data/nfcorpus
print('数据格式化完成')

In [ ]:
# 验证数据文件
import os, json
files = [
    'data/nfcorpus/train_sft.jsonl',
    'data/nfcorpus/val_sft.jsonl',
    'data/nfcorpus/test_sft.jsonl',
]
for f in files:
    if os.path.exists(f):
        with open(f) as fh:
            n = sum(1 for _ in fh)
        size = os.path.getsize(f) // 1024
        print(f'✓ {f}: {n:,} records, {size} KB')
    else:
        print(f'✗ {f}: 缺失！')

# 看第一条数据
with open('data/nfcorpus/train_sft.jsonl') as f:
    sample = json.loads(f.readline())
print(f'\n--- Sample ---')
print(f'Input length: {len(sample["input"])} chars')
print(f'Output: {sample["output"]}')
print(f'Relevance: {sample["relevance"]}')
print(f'Query ID: {sample["id"]}')
print(f'\nInput preview:\n{sample["input"][:500]}...')

---
## 2. LoRA 实验

### E21: LoRA — NFCorpus × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/lora_nfcorpus_qwen.yaml 2>&1 | tee logs/lora_nfcorpus_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.makedirs('results/nfcorpus', exist_ok=True)
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_nfcorpus_qwen.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/lora_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/lora_qwen_test.json
print('评估完成')
!cat results/nfcorpus/lora_qwen_test.json

### E22: LoRA — NFCorpus × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_nfcorpus_llama.yaml 2>&1 | tee logs/lora_nfcorpus_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_nfcorpus_llama.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/lora_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/lora_llama_test.json
print('评估完成')
!cat results/nfcorpus/lora_llama_test.json

---
## 3. Full Fine-Tuning 实验

### E23: Full FT — NFCorpus × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_nfcorpus_qwen.yaml 2>&1 | tee logs/full_nfcorpus_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_nfcorpus_qwen.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/full_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/full_qwen_test.json
print('评估完成')
!cat results/nfcorpus/full_qwen_test.json

### E24: Full FT — NFCorpus × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_nfcorpus_llama.yaml 2>&1 | tee logs/full_nfcorpus_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/full_nfcorpus_llama.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/full_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/full_llama_test.json
print('评估完成')
!cat results/nfcorpus/full_llama_test.json

---
## 4. Random Label Baseline

将训练集的 ranking output 打乱（input 不变），验证模型是否真正学到了查询-段落相关性。

### 4.0 生成 Random Label 数据

In [ ]:
import json, random
from pathlib import Path

random.seed(42)

def shuffle_labels(input_path, output_path):
    records = []
    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    outputs = [r['output'] for r in records]
    random.shuffle(outputs)
    for rec, new_out in zip(records, outputs):
        rec['output'] = new_out
        # Also update the 'text' field (full prompt with answer)
        rec['text'] = rec['input'] + new_out
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'  ✓ {output_path}: {len(records):,} records (labels shuffled)')

print('=== Generating Random Label NFCorpus Datasets ===')
shuffle_labels('data/nfcorpus/train_sft.jsonl', 'data/nfcorpus/train_sft_random.jsonl')
shuffle_labels('data/nfcorpus/val_sft.jsonl',   'data/nfcorpus/val_sft_random.jsonl')
print('\nDone. Test file is NOT shuffled (evaluate on real data).')

In [ ]:
# 验证 mismatch rate
import json

def mismatch_rate(orig_path, random_path):
    with open(orig_path) as f1, open(random_path) as f2:
        orig = [json.loads(l)['output'] for l in f1 if l.strip()]
        rand = [json.loads(l)['output'] for l in f2 if l.strip()]
    return sum(a != b for a, b in zip(orig, rand)) / len(orig)

print('Mismatch rates (should be ~1.0):')
print(f'  NFCorpus train: {mismatch_rate("data/nfcorpus/train_sft.jsonl", "data/nfcorpus/train_sft_random.jsonl"):.4f}')
print(f'  NFCorpus val:   {mismatch_rate("data/nfcorpus/val_sft.jsonl", "data/nfcorpus/val_sft_random.jsonl"):.4f}')

### E25: Random Label LoRA — NFCorpus × Qwen2.5-1.5B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_nfcorpus_qwen.yaml 2>&1 | tee logs/random_nfcorpus_qwen.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_nfcorpus_qwen.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/random_nfcorpus_qwen/predictions_test.jsonl --output results/nfcorpus/random_qwen_test.json
print('评估完成')
!cat results/nfcorpus/random_qwen_test.json

### E26: Random Label LoRA — NFCorpus × Llama-3.2-1B

In [ ]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/random_nfcorpus_llama.yaml 2>&1 | tee logs/random_nfcorpus_llama.log
print('训练完成')

In [ ]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/random_nfcorpus_llama.yaml --split test --batch_size 16 --max_new_tokens 30
!{sys.executable} -u src/evaluate/eval_rerank.py --predictions outputs/random_nfcorpus_llama/predictions_test.jsonl --output results/nfcorpus/random_llama_test.json
print('评估完成')
!cat results/nfcorpus/random_llama_test.json

---
## 5. 汇总所有 NFCorpus 结果

In [ ]:
import json, os

results = [
    ('LoRA  × Qwen',        'results/nfcorpus/lora_qwen_test.json'),
    ('LoRA  × Llama',       'results/nfcorpus/lora_llama_test.json'),
    ('Full FT × Qwen',     'results/nfcorpus/full_qwen_test.json'),
    ('Full FT × Llama',    'results/nfcorpus/full_llama_test.json'),
    ('Random × Qwen',      'results/nfcorpus/random_qwen_test.json'),
    ('Random × Llama',     'results/nfcorpus/random_llama_test.json'),
]

print(f'{"实验":<25} {"NDCG@5":>10} {"MAP@5":>10}')
print('-' * 50)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"未完成":>10}')
        continue
    with open(path) as f:
        d = json.load(f)
    ndcg = d['ndcg@5']['mean'] if isinstance(d['ndcg@5'], dict) else d['ndcg@5']
    mapk = d['map@5']['mean'] if isinstance(d['map@5'], dict) else d['map@5']
    print(f'{name:<25} {ndcg:>10.4f} {mapk:>10.4f}')